# Swin2SR classical-SR x2 — DIMER super-resolution tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/swin2sr-super-resolution-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/swin2sr-super-resolution-pipeline/blob/main/tutorials/swin2sr_super_resolution_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-caidas%2Fswin2SR--classical--sr--x2--64-ffcc4d?style=flat)](https://huggingface.co/caidas/swin2SR-classical-sr-x2-64) [![Upstream](https://img.shields.io/badge/Upstream-mv--lab%2Fswin2sr-181717?style=flat&logo=github&logoColor=white)](https://github.com/mv-lab/swin2sr) [![arXiv](https://img.shields.io/badge/arXiv-2209.11345-b31b1b.svg)](https://arxiv.org/abs/2209.11345)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** 2× single-image super-resolution (classical SR) using the pinned `caidas/swin2SR-classical-sr-x2-64` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/swin2sr_super_resolution_pipeline/pipeline.py` at revision `a00a17071d4b`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `cee1c923c6a37361c6e5650b65dcf4be821e5d52` (~48 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/1); edit the repository and regenerate rather than editing cells.

The model reconstructs a 2× larger image from one low-resolution RGB input: plausible detail is **synthesised** from what the input contains, not recovered from anywhere else. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the pinned checkpoint is used as published, and the carried pipeline module adds snapshot verification, input validation, the output contract (a uint8 RGB array of shape `(2H, 2W, 3)`), and the `psnr`, `validate_inputs` and `evaluation_report` helpers. The default sample is a synthetic image built in code together with its own high-resolution reference; the numbers it produces are demonstration (plumbing) evidence, not a production-quality or benchmark claim.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, build a synthetic high-resolution reference and its low-resolution input, validate the input into an input manifest, run the supported task, read PSNR correctly against a plain bicubic baseline and understand why a self-made reference is not a benchmark, exercise an optional BYOD path where no reference exists and the report is `not-measurable`, and export the upscaled PNG plus machine-readable outputs and provenance.

**This notebook does not demonstrate:** scales other than 2×, compressed-image or real-world (blind) restoration, denoising, artefact removal, video, face restoration, or any training. Inputs above 512 px per side are refused — tile them yourself.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. Swin2SR runs windowed attention over every input pixel, so time and memory grow with input **area**: the model card's CPU pass measured 160 s for a 1024×1024 input against 34 s for 512×512, which is why the ceiling is 512 px per side. The default 64×48 input is a fraction of a second either way. The pinned `torch==2.14.0` install and the 48 MB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python, NumPy and PIL image handling; what PSNR measures and why it is not a perceptual quality score.
- **Data:** the default sample is a deterministic 128×96 RGB image built in code and bicubic-downscaled to 64×48, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image file decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, both sides between 8 px and 512 px. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `caidas/swin2SR-classical-sr-x2-64` snapshot (~48 MB) at revision `cee1c923c6a3…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'swin2sr-super-resolution-pipeline',
    'repository_revision': 'a00a17071d4bb9356462eafcc87b7a21e30a19b7',
    'embedded_module': 'src/swin2sr_super_resolution_pipeline/pipeline.py',
    'module_sha256': '2f377c82e1dc8f2e96658989080c722ec22b16b4510f948b9c9a8ca37f92cc9b',
    'generator': 'build_notebook.py/1',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/swin2sr_super_resolution_pipeline/pipeline.py` @ `a00a17071d4b`)

This cell **is** the repository's pipeline module: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the module's, byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (currently 1: the default weights directory becomes working-directory-relative because a notebook has no `__file__`). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever this cell and the module diverge, so what you run here is what the repository tests. Nothing in this cell runs a model yet.

In [ ]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "caidas/swin2SR-classical-sr-x2-64"
MODEL_REVISION = "cee1c923c6a37361c6e5650b65dcf4be821e5d52"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "swin2sr-x2-64"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

UPSCALE = 2  # config.json "upscale": 2
# Input ceilings. Swin2SR runs windowed attention over every input pixel (patch_size 1), so activation
# memory and time grow with input area. The card pass executed up to 1024 px on CPU (160 s); the ceiling
# was set to 512 px (34 s on the reference CPU) on 2026-09-12 by the repository owner so DIMER validators
# stay responsive. See MODEL_CARD.md, Runtime.
MAX_INPUT_SIDE = 512
MIN_INPUT_SIDE = 8  # config.json "window_size": 8; the processor pads to a multiple of 8


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def psnr(pred: np.ndarray, ref: np.ndarray) -> float:
    """Peak signal-to-noise ratio in dB between two uint8 RGB arrays of identical shape.

    ``10 * log10(255**2 / MSE)`` over all channels; returns ``inf`` when the arrays are identical.
    The caller supplies the high-resolution reference; PSNR on one image is a check, not a benchmark.
    """
    pred = np.asarray(pred)
    ref = np.asarray(ref)
    if pred.shape != ref.shape:
        raise ValueError(f"shape mismatch: pred {pred.shape} vs ref {ref.shape}")
    if pred.dtype != np.uint8 or ref.dtype != np.uint8:
        raise TypeError("psnr expects uint8 arrays")
    mse = float(np.mean((pred.astype(np.float64) - ref.astype(np.float64)) ** 2))
    if mse == 0.0:
        return float("inf")
    return float(10.0 * np.log10(255.0**2 / mse))


def validate_image(image: Any) -> Image.Image:
    """Type- and size-check a caller image and return it as RGB."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_INPUT_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_INPUT_SIDE {MIN_INPUT_SIDE}")
    if max(width, height) > MAX_INPUT_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_INPUT_SIDE {MAX_INPUT_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image, or a sequence of them for the validation stage; any mode, converted to RGB",
    "image_side_px": [MIN_INPUT_SIDE, MAX_INPUT_SIDE],
    "scale": UPSCALE,
    "output": f"uint8 RGB array of shape ({UPSCALE}H, {UPSCALE}W, 3)",
    "preprocessing": (
        f"convert to RGB and pad to a multiple of the {MIN_INPUT_SIDE} px attention window; the padding "
        "is cropped back off at output scale"
    ),
}


def validate_inputs(images: Any, *, names: Sequence[str] | None = None) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Each image is routed through the public ``validate_image`` that ``upscale`` itself calls, so a
    rejection here raises exactly what ``upscale`` would; a caller that wants the finding recorded
    catches the exception and stores ``str(exc)`` under ``findings``.
    """
    batch = [images] if isinstance(images, Image.Image) else images
    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if len(batch) < 1:
        raise ValueError("at least one image is required")
    if names is not None and len(names) != len(batch):
        raise ValueError("names must have one entry per image")
    inputs = []
    for index, candidate in enumerate(batch):
        rgb = validate_image(candidate)
        inputs.append(
            {
                "id": names[index] if names else f"image-{index}",
                "mode": getattr(candidate, "mode", rgb.mode),
                "size": [rgb.width, rgb.height],
                "output_size": [rgb.width * UPSCALE, rgb.height * UPSCALE],
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "scale": UPSCALE,
        "n_images": len(inputs),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def _as_uint8_rgb(image: Any) -> np.ndarray:
    """Coerce a PIL image or an array to a uint8 RGB array; raise on anything else."""
    if isinstance(image, Image.Image):
        return np.asarray(image.convert("RGB"))
    array = np.asarray(image)
    if array.dtype != np.uint8 or array.ndim != 3 or array.shape[2] != 3:
        raise TypeError("reference must be a PIL image or a uint8 RGB array of shape (H, W, 3)")
    return array


def evaluation_report(
    result: Mapping[str, Any],
    reference: Any | None = None,
    *,
    low_resolution: Any | None = None,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference`` (the high-resolution image the output should match, same shape as
    ``result["image"]``) the report carries ``psnr`` in dB as sample-sanity evidence; pass
    ``low_resolution`` as well and the same ``psnr`` is computed for a plain bicubic upscale of the
    input, which is the only baseline worth comparing against. Without a reference the verdict is
    ``not-measurable``: a reconstruction has no intrinsic score.
    """
    output = np.asarray(result["image"])
    base = {
        "task": f"{UPSCALE}x single-image super-resolution (classical SR)",
        "score_semantics": (
            "PSNR in dB between two uint8 RGB arrays (10*log10(255**2 / MSE)); higher is closer to the "
            "reference, it is not a perceptual quality score, and the pipeline ships no threshold"
        ),
        "sample_kind": sample_kind,
        "n_images": 1,
        "scale": result.get("scale", UPSCALE),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no high-resolution reference was supplied for the evaluated image",
            "needs": (
                "high-resolution/low-resolution pairs produced by a stated degradation — a public "
                "benchmark such as Set5, Set14 or DIV2K with its own downscaling kernel — scored with "
                "psnr against a plain bicubic upscale of the same input as the baseline"
            ),
        }
    ref_array = _as_uint8_rgb(reference)
    baselines = []
    if low_resolution is not None:
        height, width = ref_array.shape[0], ref_array.shape[1]
        bicubic = _as_uint8_rgb(
            _as_pil(low_resolution).convert("RGB").resize((width, height), Image.Resampling.BICUBIC)
        )
        baselines.append(
            {
                "id": "psnr",
                "name": f"bicubic {UPSCALE}x resize of the same input",
                "value": psnr(bicubic, ref_array),
                "unit": "dB",
            }
        )
    return {
        **base,
        "metrics": [
            {
                "id": "psnr",
                "value": psnr(output, ref_array),
                "unit": "dB",
                "estimation": "single image against a caller-supplied reference, no dispersion estimate",
            }
        ],
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": (
            "one image scored against a reference the caller supplied, under the caller's own "
            "downscaling kernel; not a benchmark"
        ),
        "needs": (
            "a public benchmark set with its stated degradation kernel for any comparable PSNR claim"
        ),
    }


def _as_pil(image: Any) -> Image.Image:
    """Coerce a PIL image or a uint8 RGB array to a PIL image."""
    if isinstance(image, Image.Image):
        return image
    return Image.fromarray(_as_uint8_rgb(image))


@dataclass
class Swin2SRPipeline:
    """2x single-image super-resolution over the pinned Swin2SR classical-SR checkpoint."""

    _runner: Callable[[Image.Image], np.ndarray]
    device: str

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Swin2SRPipeline:
        import torch
        from transformers import Swin2SRForImageSuperResolution, Swin2SRImageProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        processor = Swin2SRImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = Swin2SRForImageSuperResolution.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image: Image.Image) -> np.ndarray:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                reconstruction = model(**inputs).reconstruction
            # the processor pads to a multiple of 8; crop the padding back off at output scale
            out = reconstruction[0, :, : image.height * UPSCALE, : image.width * UPSCALE]
            out = out.clamp(0.0, 1.0).mul(255.0).round().to(torch.uint8)
            return out.permute(1, 2, 0).cpu().numpy()

        return cls(runner, resolved_device)

    def upscale(self, image: Image.Image) -> dict[str, Any]:
        """Return the 2x-upscaled RGB image as a uint8 array of shape (2H, 2W, 3)."""
        rgb = validate_image(image)
        expected = (rgb.height * UPSCALE, rgb.width * UPSCALE, 3)
        result = np.asarray(self._runner(rgb))
        if result.shape != expected or result.dtype != np.uint8:
            raise RuntimeError(f"backend returned {result.shape} {result.dtype}, expected {expected} uint8")
        return {
            "image": result,
            "scale": UPSCALE,
            "input_size": (rgb.width, rgb.height),
            "output_size": (rgb.width * UPSCALE, rgb.height * UPSCALE),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `cee1c923c6a3…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Swin2SRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "swin2sr-x2-64",
  "modelId": "caidas/swin2SR-classical-sr-x2-64",
  "revision": "cee1c923c6a37361c6e5650b65dcf4be821e5d52",
  "files": [
    {
      "path": "README.md",
      "bytes": 642,
      "sha256": "9186aaf5bf94b0adb71532ad70368320e8134815fa026827c8f0e4920dc6886c"
    },
    {
      "path": "config.json",
      "bytes": 772,
      "sha256": "11179bdfd48394977f7fe6177b3a87d9c2c6fa2a67a4b3a7376229ca407f0376"
    },
    {
      "path": "model.safetensors",
      "bytes": 48460660,
      "sha256": "a515955815ab6dba3a9331d8c75db13a5166a5ffbcebf1a95ab676a5656ac4fb"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 152,
      "sha256": "cbc36266fcc93d5bc1e9ca69bcc648ae9d268918ad14cd3507216740f129cc4d"
    }
  ],
  "totalBytes": 48462226
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print({'verified_files': [entry['path'] for entry in snapshot.get('files', [])]})
pipe = Swin2SRPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Build the synthetic sample and its reference, or optional BYOD

The default sample is **synthetic** and carries its own reference: a deterministic 128×96 RGB image is built in code (horizontal colour gradient, one filled square, one diagonal stripe — sharp edges are what a super-resolver has to reconstruct), kept as the **high-resolution reference**, and bicubic-downscaled to 64×48 to become the model input. Both digests are printed. This is the same kind of gradient-and-square input the repository's smoke run used. Because you made the reference yourself, PSNR against it later is a self-consistency check of the input contract and forward pass, **not a benchmark**: real benchmarks use fixed public HR/LR pairs and a specified degradation kernel. BYOD is optional and disabled by default; when enabled, upload one image and it is upscaled as-is with no reference, so the evaluation report is `not-measurable` for it.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
HR_WIDTH, HR_HEIGHT = 128, 96

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    lr_image = Image.open(io.BytesIO(uploaded[image_name]))
    lr_image.load()
    hr_reference = None
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic high-resolution reference: no randomness, so no seed is needed.
    ramp = np.linspace(0.0, 255.0, HR_WIDTH)
    red = np.tile(ramp, (HR_HEIGHT, 1))
    green = np.tile(np.linspace(255.0, 0.0, HR_HEIGHT), (HR_WIDTH, 1)).T
    blue = np.full((HR_HEIGHT, HR_WIDTH), 96.0)
    hr_reference = Image.fromarray(np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8), mode='RGB')
    draw = ImageDraw.Draw(hr_reference)
    draw.rectangle([24, 20, 60, 56], fill=(20, 20, 20))
    draw.line([(70, 84), (118, 12)], fill=(250, 250, 250), width=5)
    # The model input is the reference bicubic-downscaled by exactly UPSCALE; the kernel choice is part of the sample.
    lr_image = hr_reference.resize((HR_WIDTH // UPSCALE, HR_HEIGHT // UPSCALE), Image.Resampling.BICUBIC)
    image_name = f'synthetic_shapes_{HR_WIDTH // UPSCALE}x{HR_HEIGHT // UPSCALE}.png'
    sample_kind = 'synthetic'

lr_sha256 = hashlib.sha256(np.asarray(lr_image.convert('RGB')).tobytes()).hexdigest()
hr_sha256 = None if hr_reference is None else hashlib.sha256(np.asarray(hr_reference).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'input_size': lr_image.size, 'input_rgb_sha256': lr_sha256, 'reference_size': None if hr_reference is None else hr_reference.size, 'reference_rgb_sha256': hr_sha256})

## 5. Validate the input → input manifest

`validate_inputs` is the pipeline's public validation stage: each image is routed through the same `validate_image` that `upscale` itself calls, so the checks — PIL type, both sides within `MIN_INPUT_SIDE`..`MAX_INPUT_SIDE` — cannot diverge between the two. It returns an **input manifest** naming the schema and ceilings, each input's identifier, observed mode, size and the output size it will produce, and the verdict, written to `outputs/swin2sr_super_resolution_input_manifest.json`. The ceilings are printed first: `UPSCALE` (2), `MIN_INPUT_SIDE` (8 px; the processor pads to a multiple of the 8-px attention window) and `MAX_INPUT_SIDE` (512 px). The 512 px ceiling was lowered from 1024 px by the repository owner on 2026-09-12 because the card-pass smoke measured 160 s for a 1024×1024 input on CPU against 34 s for 512×512, and DIMER validators must stay responsive; larger images must be tiled by the caller. To show what rejection looks like, the cell also validates a deliberately oversized image and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and padded to a multiple of 8, and the padding is cropped off at output scale; nothing else is dropped or altered.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'UPSCALE': UPSCALE, 'MIN_INPUT_SIDE': MIN_INPUT_SIDE, 'MAX_INPUT_SIDE': MAX_INPUT_SIDE}})
input_manifest = validate_inputs(lr_image, names=[image_name])
# Demonstrate rejection on an input that breaks a ceiling; the finding is recorded, not swallowed.
try:
    validate_inputs(Image.new('RGB', (MAX_INPUT_SIDE + 1, MIN_INPUT_SIDE)))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'oversized-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/swin2sr_super_resolution_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Upscale

`upscale` returns a dict with `image` (uint8 array of shape `(2H, 2W, 3)`), `scale` (2), `input_size` and `output_size` as `(width, height)`, and the model identity. The output is a deterministic reconstruction — no sampling, no seed, `torch.inference_mode` — so repeated runs on the same device and dtype give the same bytes; CPU versus CUDA kernels can differ in the last rounding step. Look for `output_size` equal to twice `input_size`.

In [ ]:
import time

started = time.perf_counter()
result = pipe.upscale(lr_image)
elapsed = time.perf_counter() - started
sr_array = result['image']
print({'scale': result['scale'], 'input_size': result['input_size'], 'output_size': result['output_size'], 'array_shape': sr_array.shape, 'dtype': str(sr_array.dtype), 'seconds': round(elapsed, 3), 'device': pipe.device})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. The repository's only metric helper is `psnr(pred, ref)`: peak signal-to-noise ratio in dB between two uint8 RGB arrays of identical shape (`10·log10(255² / MSE)`, `inf` when identical). It applies only when a high-resolution reference exists — on the synthetic default path, against the reference you generated in Section 4, so the verdict is `sample-sanity`; on BYOD none exists and the verdict is `not-measurable`, with the report naming what would make the task measurable. As a **meaningful baseline** the report also carries the same PSNR for a plain bicubic 2× resize of the input, so you can see whether the model beats interpolation on this one image. Both numbers are single-image tutorial evidence with no dispersion estimate; they depend on the downscaling kernel you chose and say nothing about photographs, compression artefacts, or the Set5/Set14/DIV2K figures reported by the upstream paper (not measured here). The report is written to `outputs/swin2sr_super_resolution_evaluation_report.json`.

In [ ]:
report = evaluation_report(result, hr_reference, low_resolution=lr_image, sample_kind=sample_kind)
with open('outputs/swin2sr_super_resolution_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No high-resolution reference exists for this input, so psnr is not computed; inspect the exported PNG instead.')
else:
    print({'model_psnr_db': report['metrics'][0]['value'], 'bicubic_baseline_db': report['baselines'][0]['value'], 'note': 'single synthetic image, self-made reference; sanity evidence, not a benchmark'})

## 8. Export outputs and provenance

The upscaled image is written as PNG (`outputs/swin2sr_super_resolution_output.png`) — the actual artifact a downstream consumer wants — and machine-readable JSON preserves the sizes and timing, the sample identity and digests, the input manifest, the evaluation report, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). No credentials are recorded.

In [ ]:
Image.fromarray(sr_array, mode='RGB').save('outputs/swin2sr_super_resolution_output.png')
payload = {
    'prediction': {key: value for key, value in result.items() if key != 'image'},
    'output_file': 'outputs/swin2sr_super_resolution_output.png',
    'output_rgb_sha256': hashlib.sha256(sr_array.tobytes()).hexdigest(),
    'seconds': round(elapsed, 3),
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'kind': sample_kind, 'name': image_name, 'input_size': list(lr_image.size), 'input_rgb_sha256': lr_sha256, 'reference_rgb_sha256': hr_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/swin2sr_super_resolution_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The upscaled image is a learned reconstruction: plausible detail synthesised from the low-resolution input, not recovered ground truth. The PSNR values shown on the synthetic path compare the model and a bicubic baseline against a reference you generated and downscaled yourself, so they measure how well each inverts *that* bicubic downscaling on one synthetic image; they are not comparable to published Set5/Set14/DIV2K numbers and must not be generalised to photographs, other kernels, compressed inputs, or other scales. On a BYOD image no reference exists and the evaluation report says `not-measurable`. Inputs above 512 px per side are refused (tile them), the model handles only 2×, and content that was never in the input cannot be recovered. The pipeline provides no denoising, artefact removal, blind restoration, video, or training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a small photograph (≤ 512 px per side) and inspect the exported PNG next to a bicubic resize; regenerate the synthetic reference with `Image.Resampling.NEAREST` or `BOX` downscaling and watch how much the model PSNR moves with the kernel alone; time `upscale` on 128×128, 256×256 and 512×512 inputs on your runtime to see the area scaling that motivated the ceiling.

## References

- Repository README: https://github.com/kurtvalcorza/swin2sr-super-resolution-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/swin2sr-super-resolution-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/swin2sr-super-resolution-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/caidas/swin2SR-classical-sr-x2-64
- Upstream code: https://github.com/mv-lab/swin2sr
- Swin2SR paper: https://arxiv.org/abs/2209.11345
- Transformers `Swin2SR` documentation: https://huggingface.co/docs/transformers/model_doc/swin2sr